This reads both CSV files, finds the rows in `J1_OmegaMid.csv` where `verified == 0`, takes the corresponding cell IDs, and extracts those cells from `cell_def.csv`. The result is saved as `cell_def_failed.csv`.

In [1]:
import pandas as pd

cells = pd.read_csv("../inputs/cell_def_v2.csv")
res = pd.read_csv("../results/J1_OmegaMid_v2.csv")

cells[cells["i"].isin(res.loc[res["verified"] == 0, "cell_id"])].to_csv(
    "../inputs/cell_def_failed_v2.csv", index=False
)

It reads `cell_def_failed.csv`, updates the specified column values, and saves it again with the same filename.

In [2]:
import pandas as pd

df = pd.read_csv("../inputs/cell_def_failed_v2.csv")
df = df.assign(
    mesh_size_lower_LG=0.08333333333,
    mesh_size_lower_cr=0.015,
)
df.to_csv("../inputs/cell_def_failed_v2.csv", index=False)

In [ ]:
import pandas as pd

base = pd.read_csv("../inputs/cell_def_origin_J1.csv").set_index("i")
failed = pd.read_csv("../inputs/cell_def_failed_J1.csv").set_index("i")

base.update(failed)
base.reset_index().to_csv("../inputs/cell_def.csv", index=False)

In [3]:
import pandas as pd

df_cell = pd.read_csv("../inputs/cell_def_failed_v3.csv")
df_j1 = pd.read_csv("../results/J1_OmegaMid_v3.csv")

assert len(df_cell) == len(df_j1), "行数が一致しません"

df_j1["cell_id"] = df_cell["i"].to_numpy()
df_j1.to_csv("../results/J1_OmegaMid_v3.csv", index=False)

In [4]:
import pandas as pd

df_j1 = pd.read_csv("../results/J1_OmegaMid_v3.csv")
df_cell = pd.read_csv("../inputs/cell_def_failed_v3.csv")

failed_ids = df_j1.loc[df_j1["verified"] == 0, "cell_id"]
df_cell[df_cell["i"].isin(failed_ids)].to_csv("../inputs/cell_def_failed_v4.csv", index=False)

In [6]:
import csv
from decimal import Decimal

src = "../inputs/cell_def_failed_v4.csv"

with open(src, newline="") as f:
    rows = list(csv.DictReader(f))
    fields = rows[0].keys()

new_rows = []
idx = 30000

for r in rows:
    x0s, x1s = r["x_inf"], r["x_sup"]
    t0s, t1s = r["theta_inf"], r["theta_sup"]

    x0, x1 = Decimal(x0s), Decimal(x1s)
    t0, t1 = Decimal(t0s), Decimal(t1s)

    xms = str((x0 + x1) / 2)
    tms = str((t0 + t1) / 2)

    for xa, xb, ta, tb in [
        (x0s, xms, t0s, tms),
        (xms, x1s, t0s, tms),
        (x0s, xms, tms, t1s),
        (xms, x1s, tms, t1s),
    ]:
        s = r.copy()
        s["i"] = str(idx)
        s["x_inf"], s["x_sup"] = xa, xb
        s["theta_inf"], s["theta_sup"] = ta, tb
        new_rows.append(s)
        idx += 1

with open(src, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(new_rows)

In [10]:
import pandas as pd

df_j1 = pd.read_csv("../results/J1_OmegaMid_v3.csv")
df_cell = pd.read_csv("../inputs/cell_def_v2.csv")

failed_ids = df_j1.loc[df_j1["verified"] == 0, "cell_id"]
df_cell[~df_cell["i"].isin(failed_ids)].to_csv("../inputs/cell_def_failed_v4_rest.csv", index=False)

In [12]:
import pandas as pd

df1 = pd.read_csv("../inputs/cell_def_failed_v4.csv")
df2 = pd.read_csv("../inputs/cell_def_failed_v4_rest.csv")

df = pd.concat([df1, df2], ignore_index=True)
df["i"] = range(1, len(df) + 1)

df.to_csv("../inputs/cell_def_v4.csv", index=False)

In [13]:
import pandas as pd

cell = pd.read_csv("../inputs/cell_def_v2.csv")
j1 = pd.read_csv("../results/J1_OmegaMid_v2.csv")
j2 = pd.read_csv("../results/J2_OmegaMid_v2.csv")

A = cell[cell["i"].isin(j1.loc[j1["verified"] == 0, "cell_id"])]
B = cell[cell["i"].isin(j2.loc[j2["verified"] == 0, "cell_id"])]

setA = set(A["i"])
setB = set(B["i"])

print("A ⊆ B:", setA <= setB)
print("B ⊆ A:", setB <= setA)
print("A = B:", setA == setB)

A ⊆ B: False
B ⊆ A: False
A = B: False


In [14]:
import pandas as pd

cell = pd.read_csv("../inputs/cell_def_v2.csv")
j1 = pd.read_csv("../results/J1_OmegaMid_v2.csv")
j2 = pd.read_csv("../results/J2_OmegaMid_v2.csv")

# A, B
ids_A = j1.loc[j1["verified"] == 0, "cell_id"]
ids_B = j2.loc[j2["verified"] == 0, "cell_id"]

A = cell[cell["i"].isin(ids_A)].copy()
B = cell[cell["i"].isin(ids_B)].copy()

# C = A ∪ B
ids_C = set(ids_A) | set(ids_B)
C = cell[cell["i"].isin(ids_C)].copy()

# cell_def_v2 から C 以外を取り，C を先頭につける
rest = cell[~cell["i"].isin(ids_C)].copy()
out = pd.concat([C, rest], ignore_index=True)

# i を振り直す
out["i"] = range(1, len(out) + 1)

# 保存
out.to_csv("../inputs/cell_def_v3.csv", index=False)

# C の範囲を表示
if len(C) > 0:
    print(f"C の部分は新しい i で 1 〜 {len(C)}")
else:
    print("C は空です")

C の部分は新しい i で 1 〜 695


In [20]:
import pandas as pd

deg = pd.read_csv("../inputs/cell_def_v3_cell_degenerate.csv")
nondeg = pd.read_csv("../inputs/cell_def_v3_non_degenerate.csv")

nondeg_1_695 = nondeg.iloc[:695]
nondeg_rest = nondeg.iloc[695:]

# 先頭に持ってくる順番：
# 1. non_degenerate の 1〜695
# 2. cell_degenerate 全体
# 3. non_degenerate の残り
df = pd.concat([nondeg_1_695, deg, nondeg_rest], ignore_index=True)

# 通し番号 i を振り直す
df["i"] = range(1, len(df) + 1)

# 新しい番号での位置を表示
a1, a2 = 1, len(nondeg_1_695)
b1, b2 = a2 + 1, a2 + len(deg)

print(f"old non_degenerate 1..695 -> new i = {a1}..{a2}")
print(f"old cell_degenerate     -> new i = {b1}..{b2}")

df.to_csv("../inputs/cell_def_v4.csv", index=False)

old non_degenerate 1..695 -> new i = 1..695
old cell_degenerate     -> new i = 696..1080


In [22]:
import pandas as pd

v2 = pd.read_csv("../inputs/cell_def_v2.csv")
failed_v2 = pd.read_csv("../inputs/cell_def_failed_v2.csv")
refined = pd.read_csv("../inputs/cell_def_failed_v2_refined.csv")

A = v2[~v2["i"].isin(failed_v2["i"])]
B = pd.concat([refined, A], ignore_index=True)

B["i"] = range(1, len(B) + 1)

print(f"old cell_def_failed_v2_refined.csv -> new i = 1..{len(refined)}")

B.to_csv("../inputs/cell_def_failed_v3_non_degenerate.csv", index=False)

old cell_def_failed_v2_refined.csv -> new i = 1..2448


In [ ]:
import pandas as pd

nondeg = pd.read_csv("../inputs/cell_def_v3_non_degenerate.csv")
deg = pd.read_csv("../inputs/cell_def_v3_degenerate.csv")
refined = pd.read_csv("../inputs/cell_def_failed_v2_refined.csv")

df = pd.concat([deg, nondeg], ignore_index=True)
df["i"] = range(1, len(df) + 1)

deg_start, deg_end = 1, len(deg)
ref_start = len(deg) + 1
ref_end = len(deg) + len(refined)

print(f"old ../inputs/cell_def_v3_degenerate.csv -> new i = {deg_start}..{deg_end}")
print(f"old cell_def_failed_v2_refined.csv      -> new i = {ref_start}..{ref_end}")

df.to_csv("../inputs/cell_def_v3.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: '../cell_def_failed_v2_refined.csv'